# Notebook 02 · El valor de cada estado y la ecuación de Bellman

**Introducción al Deep Learning — Módulo 4, Unidad 2: Aprendizaje por refuerzo (Parte I)**

En el Notebook 01 vimos el bucle agente-entorno y la política. Ahora vamos a la pregunta central del RL:

> **¿Cómo de bueno es estar en un estado?**

La respuesta es la **función de valor V(s)**: la recompensa total (descontada) que espero conseguir
a partir de ese estado si sigo mi política.

Y la herramienta para calcularla es la **ecuación de Bellman**:

$$V(s) = r(s) + \gamma \cdot V(s')$$

*El valor de un estado = recompensa inmediata + valor (descontado) del siguiente.*

Esta idea, aplicada una y otra vez, es la base de **todos** los algoritmos de RL, incluido el Q-Learning
y el DQN que veremos en la Parte II.

> 💡 Solo NumPy y Matplotlib. Todo se ejecuta en segundos.

## 1. Volvemos a montar el entorno (rejilla 4×4)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

FILAS, COLS = 4, 4
META = (3, 3)
TRAMPAS = [(1, 1), (2, 3)]
GAMMA = 0.9

ACCIONES = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
FLECHAS  = {0: "^", 1: "v", 2: "<", 3: ">"}

ESTADOS = [(f, c) for f in range(FILAS) for c in range(COLS)]

def es_terminal(s):
    return s == META or s in TRAMPAS

def paso(estado, accion):
    df, dc = ACCIONES[accion]
    f = min(max(estado[0] + df, 0), FILAS - 1)
    c = min(max(estado[1] + dc, 0), COLS - 1)
    nuevo = (f, c)
    if nuevo == META:   return nuevo, 10.0
    if nuevo in TRAMPAS: return nuevo, -10.0
    return nuevo, -1.0

print("Entorno listo:", len(ESTADOS), "estados,", len(ACCIONES), "acciones.")

## 2. La política que vamos a evaluar

Empezamos con la política más simple: **elegir al azar** entre las cuatro acciones (probabilidad 1/4 cada una).

Evaluar una política = calcular V(s) para esa política. No estamos mejorando nada todavía,
solo **midiendo** cómo de buena es cada casilla si el agente se comporta así.

In [ ]:
def politica_aleatoria(estado):
    '''Devuelve un diccionario accion -> probabilidad.'''
    return {a: 0.25 for a in ACCIONES}

## 3. Evaluación de la política con Bellman

El algoritmo es sorprendentemente simple:

1. Empezamos suponiendo que **todos los estados valen 0**.
2. Para cada estado, aplicamos Bellman: su valor es la media (ponderada por la política) de
   `recompensa + γ · valor del estado siguiente`.
3. Repetimos hasta que los valores dejen de cambiar (**converge**).

Se llama *evaluación iterativa de la política*.

In [ ]:
def evaluar_politica(politica, gamma=GAMMA, iteraciones=200, tol=1e-6, historial=False):
    V = {s: 0.0 for s in ESTADOS}
    hist = []
    for it in range(iteraciones):
        V_nuevo = dict(V)
        delta = 0.0
        for s in ESTADOS:
            if es_terminal(s):
                continue                      # los estados finales no tienen futuro
            probs = politica(s)
            valor = 0.0
            for a, p in probs.items():
                s2, r = paso(s, a)
                valor += p * (r + gamma * V[s2])   # <-- ECUACION DE BELLMAN
            V_nuevo[s] = valor
            delta = max(delta, abs(valor - V[s]))
        V = V_nuevo
        hist.append(dict(V))
        if delta < tol:
            break
    print(f"Convergió en {it+1} iteraciones (cambio máximo {delta:.2e})")
    return (V, hist) if historial else V

V_azar, hist = evaluar_politica(politica_aleatoria, historial=True)

### Vamos a verlo: el mapa de valores

In [ ]:
def mapa_valores(V, titulo):
    M = np.zeros((FILAS, COLS))
    for (f, c), v in V.items():
        M[f, c] = v
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    im = ax.imshow(M, cmap="RdYlGn")
    for (f, c), v in V.items():
        etq = "META" if (f, c) == META else ("TRAMPA" if (f, c) in TRAMPAS else f"{v:.1f}")
        ax.text(c, f, etq, ha="center", va="center", fontsize=9)
    ax.set_xticks(range(COLS)); ax.set_yticks(range(FILAS))
    ax.set_title(titulo); plt.colorbar(im, shrink=0.8); plt.show()

mapa_valores(V_azar, "V(s) con política aleatoria (γ = 0.9)")

Léelo despacio, porque aquí está toda la intuición del RL:

- Las casillas **cercanas a la meta** tienen valor alto: desde ahí es fácil ganar.
- Las casillas **junto a una trampa** tienen valor bajo: un agente que se mueve al azar cae con frecuencia.
- Y esos valores han salido **solos**, sin que nadie los escriba: solo propagando la recompensa
  hacia atrás con Bellman.

### Cómo se propaga la información

In [ ]:
seguidos = [(0, 0), (2, 2), (3, 2), (0, 1)]
plt.figure(figsize=(7, 3.5))
for s in seguidos:
    plt.plot([h[s] for h in hist[:40]], label=f"estado {s}")
plt.axhline(0, color="grey", linewidth=0.8)
plt.title("El valor de cada estado se estabiliza (convergencia)")
plt.xlabel("iteración"); plt.ylabel("V(s)"); plt.legend(); plt.show()

En la primera iteración casi todos los estados valen lo mismo. Iteración a iteración, la información
del premio (+10) y de las trampas (−10) **se propaga hacia atrás** por el mapa. Eso es Bellman en acción.

## 4. El efecto de γ

Cambiemos la paciencia del agente y veamos cómo cambia el mapa.

In [ ]:
for g in [0.2, 0.9, 0.99]:
    V_g = evaluar_politica(politica_aleatoria, gamma=g)
    mapa_valores(V_g, f"V(s) con γ = {g}")

Con **γ pequeño** el mapa es casi plano lejos de la meta: el agente es tan impaciente que un premio
a tres casillas de distancia le da igual. Con **γ grande** el gradiente hacia la meta se nota desde
mucho más lejos: el valor "viaja" por el mapa.

## 5. De los valores a una política mejor

Aquí llega el paso clave. Si conocemos V(s), podemos **mejorar la política**: en cada casilla,
elegir la acción que lleva al estado con mejor `recompensa + γ · V(s')`.

Esto es exactamente lo que hace el valor **Q(s, a)**: el valor de tomar una acción concreta.

In [ ]:
def Q_de(s, a, V, gamma=GAMMA):
    s2, r = paso(s, a)
    return r + gamma * V[s2]

def politica_voraz(V):
    pol = {}
    for s in ESTADOS:
        if es_terminal(s):
            continue
        pol[s] = max(ACCIONES, key=lambda a: Q_de(s, a, V))
    return pol

pol_mejor = politica_voraz(V_azar)

print("Q(s,a) para el estado (0,0):")
for a in ACCIONES:
    print(f"  {FLECHAS[a]}  Q = {Q_de((0,0), a, V_azar):+7.3f}")
print("\nMejor acción en (0,0):", FLECHAS[pol_mejor[(0, 0)]])

### La política mejorada, dibujada sobre el mapa

In [ ]:
M = np.zeros((FILAS, COLS))
for (f, c), v in V_azar.items(): M[f, c] = v
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.imshow(M, cmap="RdYlGn")
for s in ESTADOS:
    f, c = s
    if s == META:        ax.text(c, f, "META", ha="center", va="center", fontsize=8)
    elif s in TRAMPAS:   ax.text(c, f, "X", ha="center", va="center", fontsize=13)
    else:                ax.text(c, f, FLECHAS[pol_mejor[s]], ha="center", va="center", fontsize=18)
ax.set_xticks(range(COLS)); ax.set_yticks(range(FILAS))
ax.set_title("Política mejorada a partir de V(s)")
plt.show()

Cada flecha es la mejor acción según los valores calculados. **El agente ya sabe hacia dónde ir**,
y no se lo hemos dicho nosotros: lo ha deducido de las recompensas.

### ¿Cuánto mejora en la práctica?

In [ ]:
def jugar(elige_accion, max_pasos=50):
    s, total = (0, 0), 0.0
    for _ in range(max_pasos):
        s2, r = paso(s, elige_accion(s))
        total += r; s = s2
        if es_terminal(s): break
    return total

azar   = [jugar(lambda s: np.random.randint(4)) for _ in range(500)]
voraz  = [jugar(lambda s: pol_mejor[s]) for _ in range(500)]
print(f"Política aleatoria : {np.mean(azar):+.2f}")
print(f"Política de V(s)   : {np.mean(voraz):+.2f}")

## Experimenta tú

1. Repite la evaluación partiendo de `pol_mejor` (conviértela a diccionario de probabilidades) y vuelve a
   mejorarla. Repitiendo evaluación → mejora se llega a la **política óptima**: eso es *iteración de política*.
2. Cambia el castigo por paso de `-1` a `-0.1` y mira cómo cambian las flechas.
3. Añade una casilla con recompensa `+3` intermedia. ¿El agente se desvía a recogerla? ¿Depende de γ?

## Resumen

- **V(s)** mide cómo de bueno es un estado; **Q(s,a)**, cómo de buena es una acción en ese estado.
- La **ecuación de Bellman** relaciona el valor de un estado con el del siguiente, y aplicada
  iterativamente permite calcular V(s) para una política (*evaluación de la política*).
- Con V(s) podemos **mejorar la política**: elegir en cada estado la acción de mayor Q.
- **γ** controla cuánto se propaga la recompensa por el mapa: la paciencia del agente.

⚠️ Aquí hemos hecho trampa en un punto: conocíamos el entorno (la función `paso`). En un problema real
el agente **no lo conoce** y tiene que estimar Q **actuando**. Eso es el **Q-Learning**, y lo veremos
en la Parte II, junto con las **Deep Q-Networks (DQN)**, que sustituyen la tabla Q por una red neuronal.